# Práctica 01 — Qué es el NLP

**Curso:** NLP Clásico con Python · **Clase 01** · **Tiempo estimado:** 45-60 minutos
**Notebook de clase:** `notebooks_clase/clase01_que_es_el_nlp.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una o varias celdas con `# TODO`: complétalas.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te indica qué revisar.
- Todos los textos están en la celda de datos que sigue al setup. Son cortos y están escritos para el curso (*ejemplo ilustrativo*).
- Las **soluciones** están al final del notebook, en una sección separada. **No las abras antes de intentarlo.**
- Ejecuta las celdas en orden. No hace falta ninguna librería aparte de Python.

---
## Celda 0 — Setup y datos

In [ ]:
import re
from collections import Counter

print("Librerías cargadas correctamente.")

In [ ]:
# datos-es
RAIZ_ES = "habl"
TER_ES = {"pres": ["o", "as", "a", "amos", "áis", "an"], "imp": ["aba", "abas", "aba", "ábamos", "abais", "aban"], "fut": ["aré", "arás", "ará", "aremos", "aréis", "arán"]}
RAIZ_IT = "parl"
TER_IT = {"pres": ["o", "i", "a", "iamo", "ate", "ano"], "imp": ["avo", "avi", "ava", "avamo", "avate", "avano"], "fut": ["erò", "erai", "erà", "eremo", "erete", "eranno"]}

FRASES_AMBIGUAS = {
    1: "Vi al hombre con el telescopio",
    2: "Me senté en el banco del parque",
    3: "Ho comprato una pesca al mercato",
    4: "Ho visto la ragazza con il binocolo",
}

POSITIVAS = {"spa": {"bueno", "buena", "excelente"}, "ita": {"buono", "buona", "ottimo"}}
NEGATIVAS = {"spa": {"malo", "mala", "horrible"}, "ita": {"cattivo", "pessimo", "orribile"}}
NEGADORES = {"spa": {"no", "nunca"}, "ita": {"non", "mai"}}
CASOS_NEGACION = [
    ("spa", "el producto es bueno", "pos"),
    ("spa", "el producto no es bueno", "neg"),
    ("spa", "nunca fue malo", "pos"),
    ("spa", "servicio horrible", "neg"),
    ("spa", "todo normal", None),
    ("ita", "il servizio è buono", "pos"),
    ("ita", "il servizio non è buono", "neg"),
    ("ita", "non è orribile", "pos"),
    ("ita", "prodotto orribile", "neg"),
    ("ita", "tutto normale", None),
]

PERFILES = {
    "spa": {"el", "los", "las", "de", "que", "y", "en", "es", "por", "con", "para", "pero", "mi", "muy", "no", "hay", "está", "estoy", "son", "se", "nada", "cuando", "esta", "tienda", "tiene", "necesito", "otra", "quiero"},
    "ita": {"il", "lo", "i", "gli", "le", "di", "che", "e", "in", "è", "per", "con", "ma", "mio", "molto", "non", "sono", "si", "nel", "niente", "quando", "questo", "c", "ho", "ne", "serve", "altra", "vorrei", "hanno"},
}
FRASES_IDIOMA = [
    ("El servicio es muy bueno y la entrega fue rápida", "spa"),
    ("No hay nada en mi cuenta y necesito ayuda", "spa"),
    ("Los productos de esta tienda son caros pero de buena calidad", "spa"),
    ("La aplicación se cierra cuando intento pagar con la tarjeta", "spa"),
    ("Estoy muy contento con el envío y con la atención", "spa"),
    ("Il servizio è molto buono e la consegna è stata rapida", "ita"),
    ("Non c'è niente nel mio conto e ho bisogno di aiuto", "ita"),
    ("I prodotti di questo negozio sono cari ma di buona qualità", "ita"),
    ("L'applicazione si chiude quando provo a pagare con la carta", "ita"),
    ("Sono molto contento della spedizione e dell'assistenza", "ita"),
]

CATEGORIAS = {
    "spa": {"fact": {"cobraron", "tarjeta", "reembolso", "factura", "importe", "cobro", "pago"},
            "env": {"paquete", "pedido", "llegó", "llegado", "envíen", "retraso", "entrega"},
            "tec": {"aplicación", "app", "error", "contraseña", "conectar", "funciona", "cuenta"}},
    "ita": {"fact": {"addebitato", "carta", "rimborso", "fattura", "importo"},
            "env": {"pacco", "consegna", "ritardo", "arrivato", "ordine", "spedizione"},
            "tec": {"app", "errore", "password", "funziona", "account", "collegarmi"}},
}
TICKETS_PRUEBA = [
    ("Me cobraron dos veces con la tarjeta y quiero un reembolso", "spa", "fact"),
    ("El paquete no ha llegado y el pedido lleva diez días de retraso", "spa", "env"),
    ("La aplicación muestra un error y no puedo conectar con mi cuenta", "spa", "tec"),
    ("La factura tiene un importe equivocado, necesito otra", "spa", "fact"),
    ("Mi pedido llegó roto y quiero que me envíen otro paquete", "spa", "env"),
    ("No funciona la contraseña nueva, la app da error", "spa", "tec"),
    ("Mi hanno addebitato due volte sulla carta e voglio il rimborso", "ita", "fact"),
    ("Il pacco non è arrivato e la consegna è in ritardo di dieci giorni", "ita", "env"),
    ("L'app mostra un errore e non riesco a collegarmi al mio account", "ita", "tec"),
    ("La fattura ha un importo sbagliato, ne serve un'altra", "ita", "fact"),
    ("Il mio ordine è arrivato rotto e vorrei un altro pacco", "ita", "env"),
    ("La nuova password non funziona, l'app dà errore", "ita", "tec"),
]

## Ejercicio 1 — Contar las formas de un verbo

Con las terminaciones de `TER_ES` (verbo `hablar`) y `TER_IT` (verbo `parlare`) para tres tiempos (presente, imperfecto y futuro, seis personas cada uno), calcula:

- `formas_es`, `formas_it`: la lista de las 18 formas de cada idioma (raíz + terminación),
- `distintas_es`, `distintas_it`: cuántas formas **distintas** hay,
- `repetidas_es`: el **conjunto** de formas que aparecen en más de un hueco (en italiano no hay ninguna).

Antes de programar, comprueba a mano si en español coincide alguna forma entre las 18.

In [ ]:
formas_es = None      # TODO: lista con raíz + terminación para los tres tiempos
formas_it = None      # TODO
distintas_es = None   # TODO
distintas_it = None   # TODO
repetidas_es = None   # TODO: conjunto de formas con más de una aparición

# Comprobación automática
assert all(v is not None for v in (formas_es, formas_it, distintas_es, distintas_it, repetidas_es)), "Completa las líneas marcadas con TODO"
assert len(formas_es) == 18 and len(formas_it) == 18, "cada idioma tiene 3 tiempos × 6 personas = 18 formas"
assert distintas_es == 17 and distintas_it == 18, "en español una forma se repite en el imperfecto; en italiano no se repite ninguna"
assert repetidas_es == {RAIZ_ES + "aba"}, "la forma repetida es la primera y la tercera persona del imperfecto"
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Tipos de ambigüedad

En `FRASES_AMBIGUAS` tienes cuatro frases. Clasifica cada una según su tipo de ambigüedad con el código: `"lex"` (léxica: una palabra con varios sentidos) o `"syn"` (sintáctica: dos formas de agrupar las palabras). Rellena el diccionario `tipo_ambiguedad` (clave = número de frase).

In [ ]:
tipo_ambiguedad = {1: None, 2: None, 3: None, 4: None}      # TODO: "lex" o "syn" para cada frase

# Comprobación automática
assert all(v is not None for v in tipo_ambiguedad.values()), "Completa las líneas marcadas con TODO"
assert set(tipo_ambiguedad.values()) <= {"lex", "syn"}, "usa solo los códigos lex y syn"
assert tipo_ambiguedad[1] == "syn" and tipo_ambiguedad[4] == "syn", "«con el telescopio» / «con il binocolo»: ¿a qué palabra se une el grupo?"
assert tipo_ambiguedad[2] == "lex" and tipo_ambiguedad[3] == "lex", "banco y pesca tienen varios sentidos"
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Un clasificador de reglas con negación

Completa la función `clasificar` para que un negador invierta la polaridad de la palabra con carga que aparece hasta `ventana` posiciones después. La función devuelve `"pos"`, `"neg"` o `None` si hay empate (o no hay ninguna palabra con carga). Se prueba con diez frases en los dos idiomas (`CASOS_NEGACION`).

In [ ]:
def clasificar(texto, idioma, ventana=2):
    palabras = re.findall(r"\w+", texto.lower())
    puntos = 0
    for i, palabra in enumerate(palabras):
        valor = 1 if palabra in POSITIVAS[idioma] else -1 if palabra in NEGATIVAS[idioma] else 0
        hay_negador = None      # TODO: ¿hay un negador en las `ventana` palabras anteriores?
        if hay_negador:
            valor = -valor
        puntos += valor
    return "pos" if puntos > 0 else "neg" if puntos < 0 else None

# Comprobación automática
for idioma, texto, esperado in CASOS_NEGACION:
    assert clasificar(texto, idioma) == esperado, f"revisa la negación en: {texto!r} (esperado {esperado})"
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Ordenar una pipeline

Una tienda quiere clasificar sus tickets de soporte por categoría con un modelo de aprendizaje. Estos son los pasos, desordenados:

`PASOS = ["representar", "evaluar", "limpiar", "entrenar", "adquirir", "tokenizar", "dividir"]`

Escribe en `orden` una lista con los siete pasos en un orden **válido**. Hay más de un orden posible; la comprobación solo exige las dependencias: primero se adquieren los datos, se limpian, se tokenizan; el conjunto de prueba se separa **antes de representar** (para no usar información de la prueba al construir el vocabulario); se representa antes de entrenar y se entrena antes de evaluar.

In [ ]:
PASOS = ["representar", "evaluar", "limpiar", "entrenar", "adquirir", "tokenizar", "dividir"]
orden = None      # TODO: lista con los siete pasos en un orden válido

# Comprobación automática
assert orden is not None, "Completa las líneas marcadas con TODO"
assert sorted(orden) == sorted(PASOS), "usa cada paso exactamente una vez"
RESTRICCIONES = [("adquirir", "limpiar"), ("limpiar", "tokenizar"), ("tokenizar", "representar"),
                 ("dividir", "representar"), ("representar", "entrenar"), ("entrenar", "evaluar")]
for antes, despues in RESTRICCIONES:
    assert orden.index(antes) < orden.index(despues), f"'{antes}' debe ir antes de '{despues}'"
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — Un detector de italiano y español

`PERFILES` guarda, para cada idioma (`"spa"`, `"ita"`), un conjunto de palabras muy frecuentes (un perfil de stopwords). Completa `detectar_idioma`: tokeniza el texto en minúsculas con `re.findall(r"\w+", ...)`, cuenta cuántas palabras están en cada perfil y devuelve el idioma con más coincidencias, o `None` si hay empate (por ejemplo, un texto vacío). Se prueba con diez frases (`FRASES_IDIOMA`).

> **Nota:** los perfiles y las palabras clave se han elegido a mano para este mini ejercicio y se prueban sobre las mismas frases que ves. Sirve para practicar la lógica, **no como evaluación real**: un detector serio se evalúa con datos separados de los que se usaron para construirlo (lo haremos a partir de la clase 5).

In [ ]:
def detectar_idioma(texto):
    palabras = re.findall(r"\w+", texto.lower())
    aciertos_spa = None      # TODO: cuántas palabras están en el perfil "spa"
    aciertos_ita = None      # TODO: cuántas palabras están en el perfil "ita"
    if aciertos_spa is None or aciertos_ita is None:
        return None
    return "spa" if aciertos_spa > aciertos_ita else "ita" if aciertos_ita > aciertos_spa else None

# Comprobación automática
assert detectar_idioma("") is None, "un texto vacío no tiene idioma"
aciertos = sum(detectar_idioma(texto) == idioma for texto, idioma in FRASES_IDIOMA)
assert aciertos == len(FRASES_IDIOMA), f"acierta {aciertos} de {len(FRASES_IDIOMA)}: revisa el recuento"
print(f"Ejercicio 5 correcto ✔ ({aciertos} de {len(FRASES_IDIOMA)} frases)")

---
## Reto final — Idioma y categoría de un ticket

Integra lo visto: escribe `analizar_ticket(texto)`, que devuelve una pareja `(idioma, categoria)`.

- El idioma se decide con los perfiles de `PERFILES` (como en el ejercicio 5).
- La categoría (`"fact"` = facturación, `"env"` = envío, `"tec"` = problema técnico) se decide con las palabras clave de `CATEGORIAS[idioma]`: gana la categoría con más palabras clave presentes; `None` si hay empate.

Se evalúa sobre doce tickets de prueba (`TICKETS_PRUEBA`, seis por idioma; *ejemplo ilustrativo*) y se exige una exactitud mínima de 11 de 12 tanto en el idioma como en la categoría.

> **Nota:** los perfiles y las palabras clave se han elegido a mano para este mini ejercicio y se prueban sobre las mismas frases que ves. Sirve para practicar la lógica, **no como evaluación real**: un detector serio se evalúa con datos separados de los que se usaron para construirlo (lo haremos a partir de la clase 5).

In [ ]:
def analizar_ticket(texto):
    palabras = re.findall(r"\w+", texto.lower())
    # TODO 1: decide el idioma contando palabras en cada perfil
    idioma = None
    # TODO 2: con las palabras clave de CATEGORIAS[idioma], elige la categoría con más coincidencias
    categoria = None
    return idioma, categoria

# Comprobación automática
resultados_idioma = []
resultados_categoria = []
for texto, idioma_real, categoria_real in TICKETS_PRUEBA:
    idioma, categoria = analizar_ticket(texto)
    resultados_idioma.append(idioma == idioma_real)
    resultados_categoria.append(categoria == categoria_real)
assert sum(resultados_idioma) >= 11, f"idioma: {sum(resultados_idioma)} de 12"
assert sum(resultados_categoria) >= 11, f"categoría: {sum(resultados_categoria)} de 12"
print(f"Reto final correcto ✔ (idioma {sum(resultados_idioma)}/12, categoría {sum(resultados_categoria)}/12)")

**Tus conclusiones (escribe 3-4 frases):** ¿qué falla en tu detector de idioma con textos muy cortos o con palabras compartidas entre italiano y español? ¿Qué le pasaría a las reglas de categoría con un ticket que mezcla dos problemas, o con una palabra mal escrita? ¿En qué punto de la pipeline arreglarías cada fallo?

*(Tu respuesta aquí.)*

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
formas_es = [RAIZ_ES + t for lista in TER_ES.values() for t in lista]
formas_it = [RAIZ_IT + t for lista in TER_IT.values() for t in lista]
distintas_es = len(set(formas_es))
distintas_it = len(set(formas_it))
repetidas_es = {f for f, n in Counter(formas_es).items() if n > 1}

### Solución del ejercicio 2

In [ ]:
tipo_ambiguedad = {1: "syn", 2: "lex", 3: "lex", 4: "syn"}

assert all(v is not None for v in tipo_ambiguedad.values()), "Completa las líneas marcadas con TODO"

### Solución del ejercicio 3

In [ ]:
def clasificar(texto, idioma, ventana=2):
    palabras = re.findall(r"\w+", texto.lower())
    puntos = 0
    for i, palabra in enumerate(palabras):
        valor = 1 if palabra in POSITIVAS[idioma] else -1 if palabra in NEGATIVAS[idioma] else 0
        hay_negador = any(p in NEGADORES[idioma] for p in palabras[max(0, i - ventana):i])
        if hay_negador:
            valor = -valor
        puntos += valor
    return "pos" if puntos > 0 else "neg" if puntos < 0 else None

for idioma, texto, esperado in CASOS_NEGACION:
    assert clasificar(texto, idioma) == esperado, f"revisa la negación en: {texto!r} (esperado {esperado})"

### Solución del ejercicio 4

In [ ]:
PASOS = ["representar", "evaluar", "limpiar", "entrenar", "adquirir", "tokenizar", "dividir"]
orden = ["adquirir", "limpiar", "tokenizar", "dividir", "representar", "entrenar", "evaluar"]

assert orden is not None, "Completa las líneas marcadas con TODO"

### Solución del ejercicio 5

In [ ]:
def detectar_idioma(texto):
    palabras = re.findall(r"\w+", texto.lower())
    aciertos_spa = sum(p in PERFILES["spa"] for p in palabras)
    aciertos_ita = sum(p in PERFILES["ita"] for p in palabras)
    if aciertos_spa is None or aciertos_ita is None:
        return None
    return "spa" if aciertos_spa > aciertos_ita else "ita" if aciertos_ita > aciertos_spa else None

aciertos = sum(detectar_idioma(texto) == idioma for texto, idioma in FRASES_IDIOMA)

### Solución del reto final

In [ ]:
def analizar_ticket(texto):
    palabras = re.findall(r"\w+", texto.lower())
    puntos_idioma = {i: sum(p in perfil for p in palabras) for i, perfil in PERFILES.items()}
    mejor = max(puntos_idioma.values())
    ganadores = [i for i, n in puntos_idioma.items() if n == mejor]
    idioma = ganadores[0] if len(ganadores) == 1 else None
    if idioma is None:
        return None, None
    puntos_cat = {c: sum(p in claves for p in palabras) for c, claves in CATEGORIAS[idioma].items()}
    mejor_cat = max(puntos_cat.values())
    ganadoras = [c for c, n in puntos_cat.items() if n == mejor_cat]
    categoria = ganadoras[0] if mejor_cat > 0 and len(ganadoras) == 1 else None
    return idioma, categoria

resultados_idioma = []
resultados_categoria = []
for texto, idioma_real, categoria_real in TICKETS_PRUEBA:
    idioma, categoria = analizar_ticket(texto)
    resultados_idioma.append(idioma == idioma_real)
    resultados_categoria.append(categoria == categoria_real)
assert sum(resultados_idioma) >= 11, f"idioma: {sum(resultados_idioma)} de 12"
assert sum(resultados_categoria) >= 11, f"categoría: {sum(resultados_categoria)} de 12"
print(f"Reto final correcto ✔ (idioma {sum(resultados_idioma)}/12, categoría {sum(resultados_categoria)}/12)")

---
## Autoevaluación de la clase

Responde con honestidad (una o dos frases cada una):

1. ¿Qué ejercicio te ha costado más y por qué?
2. ¿Qué concepto de la clase no te ha quedado claro (pipeline, tipos de ambigüedad, morfología, clíticos, clásico frente a neuronal)?
3. ¿Qué cambiarías o repetirías la próxima vez?
4. ¿Cuánto tiempo has dedicado en total?

*(Tus respuestas aquí.)*